# Attention Mechanisms for GPT-2

This notebook provides a ground-up walkthrough of the attention mechanism that lies at the heart of GPT-2 and all decoder-only transformers.

**What we will cover:**

1. **Raw dot-product attention** -- computing Q, K, V from scratch and understanding each matrix operation.
2. **Scaled dot-product attention** -- why dividing by sqrt(d_k) is critical and what happens without it.
3. **Causal (autoregressive) masking** -- ensuring the model cannot peek at future tokens.
4. **Multi-head attention** -- splitting representation space into parallel heads and recombining.
5. **Visualizations** -- heatmaps, histograms, and side-by-side comparisons that build intuition.
6. **Ablations** -- systematically removing components to see what breaks and why.

All code is self-contained. We use tiny dimensions (`embed_dim=64`, `n_heads=4`, `seq_len=16`) so every cell runs in well under 30 seconds.

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Tiny dimensions for fast experimentation
EMBED_DIM = 64
N_HEADS = 4
HEAD_DIM = EMBED_DIM // N_HEADS  # 16
SEQ_LEN = 16
BATCH_SIZE = 2

# Token labels for visualization (we will use the first 8 or all 16 as needed)
TOKEN_LABELS_8 = ["The", "cat", "sat", "on", "the", "mat", ".", "<end>"]
TOKEN_LABELS_16 = [
    "The", "cat", "sat", "on", "the", "mat", ".", "<end>",
    "It", "was", "a", "nice", "warm", "sunny", "day", "."
]

print(f"embed_dim={EMBED_DIM}, n_heads={N_HEADS}, head_dim={HEAD_DIM}")
print(f"seq_len={SEQ_LEN}, batch_size={BATCH_SIZE}")
print(f"Device: cpu (tiny dims -- no GPU needed)")

---
## 2. Raw Dot-Product Attention from Scratch

The fundamental attention computation consists of three steps:

1. **Score**: compute how much each query token "cares about" each key token via a dot product.
2. **Normalize**: apply softmax so scores become a probability distribution.
3. **Aggregate**: use those probabilities to take a weighted sum of the value vectors.

Mathematically: $\text{Attention}(Q, K, V) = \text{softmax}(QK^T) V$

In [ ]:
# Create random Q, K, V matrices
# Shape: (batch_size, seq_len, embed_dim)
Q = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
K = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
V = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

In [ ]:
# Step 1: Compute raw attention scores = Q @ K^T
# (B, T, C) @ (B, C, T) -> (B, T, T)
raw_scores = torch.bmm(Q, K.transpose(1, 2))

print(f"Raw scores shape: {raw_scores.shape}")
print(f"Score[0, 0, :5] (query 0 vs first 5 keys): {raw_scores[0, 0, :5].tolist()}")
print(f"\nScore statistics:")
print(f"  Mean:  {raw_scores.mean().item():.3f}")
print(f"  Std:   {raw_scores.std().item():.3f}")
print(f"  Min:   {raw_scores.min().item():.3f}")
print(f"  Max:   {raw_scores.max().item():.3f}")

# Step 2: Apply softmax to get attention weights
raw_weights = F.softmax(raw_scores, dim=-1)

print(f"\nAttention weights shape: {raw_weights.shape}")
print(f"Weights[0, 0, :5]: {raw_weights[0, 0, :5].tolist()}")
print(f"Sum of weights for query 0: {raw_weights[0, 0].sum().item():.6f} (should be 1.0)")

# Step 3: Compute output = weights @ V
# (B, T, T) @ (B, T, C) -> (B, T, C)
raw_output = torch.bmm(raw_weights, V)

print(f"\nShape summary:")
print(f"  Q:       {Q.shape}")
print(f"  K:       {K.shape}")
print(f"  V:       {V.shape}")
print(f"  Scores:  {raw_scores.shape}  (Q @ K^T)")
print(f"  Weights: {raw_weights.shape}  (softmax(scores))")
print(f"  Output:  {raw_output.shape}  (weights @ V)")

**Observation**: The output has the same shape as the input queries -- each query position gets a new vector that is a weighted combination of all value vectors. The weights are determined by how similar each query is to each key.

---
## 3. Why Scaling by sqrt(d_k) Matters

When `d_k` (the dimension of the key vectors) is large, the dot products $QK^T$ grow in magnitude. This pushes the softmax inputs into regions where the gradients are extremely small (the softmax saturates), making training unstable.

The fix from "Attention Is All You Need": divide by $\sqrt{d_k}$ before softmax.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
# Demonstrate the problem: as d_k grows, dot-product variance grows linearly
dims_to_test = [16, 64, 256, 1024]

print("d_k | Score std (unscaled) | Score std (scaled)")
print("-" * 55)

for d_k in dims_to_test:
    q = torch.randn(1, 8, d_k)
    k = torch.randn(1, 8, d_k)
    scores_unscaled = torch.bmm(q, k.transpose(1, 2))
    scores_scaled = scores_unscaled / (d_k ** 0.5)
    print(f"{d_k:>4d} | {scores_unscaled.std().item():>20.3f} | {scores_scaled.std().item():>17.3f}")

In [ ]:
# Compare attention weight distributions: with and without scaling
d_k = EMBED_DIM
scores = torch.bmm(Q, K.transpose(1, 2))

weights_unscaled = F.softmax(scores, dim=-1)
weights_scaled = F.softmax(scores / (d_k ** 0.5), dim=-1)

print("Without scaling:")
print(f"  Max weight:     {weights_unscaled.max().item():.4f}")
print(f"  Min weight:     {weights_unscaled.min().item():.6f}")
print(f"  Entropy (avg):  {-(weights_unscaled * weights_unscaled.clamp(min=1e-9).log()).sum(-1).mean().item():.4f}")

print("\nWith scaling (/ sqrt(d_k)):")
print(f"  Max weight:     {weights_scaled.max().item():.4f}")
print(f"  Min weight:     {weights_scaled.min().item():.6f}")
print(f"  Entropy (avg):  {-(weights_scaled * weights_scaled.clamp(min=1e-9).log()).sum(-1).mean().item():.4f}")

print(f"\nUniform entropy for {SEQ_LEN} tokens: {np.log(SEQ_LEN):.4f}")

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention.
    
    Args:
        Q: (B, T, d_k) query matrix
        K: (B, T, d_k) key matrix
        V: (B, T, d_v) value matrix
        mask: optional (T, T) or (B, T, T) mask, True where attention is allowed
    
    Returns:
        output: (B, T, d_v)
        weights: (B, T, T)
    """
    d_k = Q.shape[-1]
    scores = torch.bmm(Q, K.transpose(1, 2)) / (d_k ** 0.5)
    
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    
    weights = F.softmax(scores, dim=-1)
    output = torch.bmm(weights, V)
    return output, weights


# Quick test
out, w = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {out.shape}")
print(f"Weights shape: {w.shape}")
print(f"Weights sum per query: {w[0].sum(dim=-1)[:4].tolist()} (all ~1.0)")

**Key point**: Scaling keeps the softmax in its "useful" regime where gradients flow. Without scaling, the softmax becomes nearly one-hot, which kills gradient flow and makes the model unable to learn distributed attention patterns.

---
## 4. Causal Masking

GPT-2 is an *autoregressive* model: when predicting token $t$, it should only have access to tokens $0, 1, \ldots, t-1$ (and $t$ itself). We enforce this with a **causal mask** -- an upper-triangular matrix of `-inf` values that, after softmax, zeros out attention to future positions.

In [ ]:
# Create the causal mask
# True = allowed to attend, False = blocked
causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool))

print("Causal mask (1 = attend, 0 = block):")
print(causal_mask[:8, :8].int().numpy())

In [ ]:
# Apply causal mask to attention scores
scores = torch.bmm(Q, K.transpose(1, 2)) / (EMBED_DIM ** 0.5)
print(f"Scores before masking [0, :4, :4]:")
print(scores[0, :4, :4].detach().numpy().round(3))

# Apply mask: set future positions to -inf
masked_scores = scores.masked_fill(~causal_mask.unsqueeze(0), float('-inf'))
print(f"\nScores after masking [0, :4, :4]:")
print(masked_scores[0, :4, :4].detach().numpy().round(3))

# Softmax turns -inf into 0
causal_weights = F.softmax(masked_scores, dim=-1)
print(f"\nAttention weights after masking [0, :4, :4]:")
print(causal_weights[0, :4, :4].detach().numpy().round(4))

# Verify: each position only attends to current and previous positions
print("\nChecking that future attention weights are zero...")
for t in range(min(6, SEQ_LEN)):
    future_weight = causal_weights[0, t, t+1:].sum().item()
    past_weight = causal_weights[0, t, :t+1].sum().item()
    print(f"  Position {t:2d}: past+current weight = {past_weight:.6f}, future weight = {future_weight:.6f}")

print("\nAll future weights are zero -- causal masking is correct.")

In [ ]:
# --- Visualizing the Causal Mask ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. The mask itself
ax = axes[0]
im = ax.imshow(causal_mask[:8, :8].int().numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_title('Causal Mask (8x8)', fontsize=13)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(TOKEN_LABELS_8, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# 2. Scores before masking
ax = axes[1]
im = ax.imshow(scores[0, :8, :8].detach().numpy(), cmap='RdBu_r', aspect='auto')
ax.set_title('Scores (before mask)', fontsize=13)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(TOKEN_LABELS_8, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# 3. Weights after masking
ax = axes[2]
im = ax.imshow(causal_weights[0, :8, :8].detach().numpy(), cmap='viridis', aspect='auto')
ax.set_title('Weights (after causal mask)', fontsize=13)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(TOKEN_LABELS_8, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

**Analysis**: The lower-triangular structure is clearly visible in the weight heatmap. The first token ("The") can only attend to itself -- it gets 100% of its own attention. Later tokens spread their attention across all preceding positions. This is exactly the autoregressive property GPT-2 needs for language generation.

---
## 5. Multi-Head Attention

Instead of performing a single attention function, multi-head attention:

1. **Projects** Q, K, V into `n_heads` separate subspaces (each of dimension `head_dim = embed_dim / n_heads`).
2. **Attends** independently in each subspace.
3. **Concatenates** all head outputs.
4. **Projects** back to `embed_dim` with a final linear layer.

This allows different heads to learn different types of relationships (e.g., one head for syntax, another for coreference).

In [ ]:
# Manual multi-head attention step by step
torch.manual_seed(42)

# Input: (B, T, C)
x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

# Projection matrices
W_q = nn.Linear(EMBED_DIM, EMBED_DIM, bias=False)
W_k = nn.Linear(EMBED_DIM, EMBED_DIM, bias=False)
W_v = nn.Linear(EMBED_DIM, EMBED_DIM, bias=False)
W_o = nn.Linear(EMBED_DIM, EMBED_DIM, bias=False)

# Project
q = W_q(x)  # (B, T, C)
k = W_k(x)
v = W_v(x)

print(f"After projection: q={q.shape}, k={k.shape}, v={v.shape}")

# Reshape into multiple heads: (B, T, C) -> (B, T, n_heads, head_dim) -> (B, n_heads, T, head_dim)
B, T, C = q.shape

q_heads = q.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)  # (B, n_heads, T, head_dim)
k_heads = k.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
v_heads = v.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)

print(f"q_heads shape: {q_heads.shape}  -- (batch, n_heads, seq_len, head_dim)")
print(f"k_heads shape: {k_heads.shape}")
print(f"v_heads shape: {v_heads.shape}")

In [ ]:
# Scaled dot-product attention per head
# scores: (B, n_heads, T, head_dim) @ (B, n_heads, head_dim, T) -> (B, n_heads, T, T)
scores_mh = torch.matmul(q_heads, k_heads.transpose(-2, -1)) / (HEAD_DIM ** 0.5)

# Apply causal mask
causal_mask_4d = causal_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T) for broadcasting
scores_mh = scores_mh.masked_fill(~causal_mask_4d, float('-inf'))

# Softmax
weights_mh = F.softmax(scores_mh, dim=-1)

# Weighted sum of values
# (B, n_heads, T, T) @ (B, n_heads, T, head_dim) -> (B, n_heads, T, head_dim)
attn_output = torch.matmul(weights_mh, v_heads)

print(f"Per-head scores:  {scores_mh.shape}")
print(f"Per-head weights: {weights_mh.shape}")
print(f"Per-head output:  {attn_output.shape}")

# Concatenate heads and apply output projection
# (B, n_heads, T, head_dim) -> (B, T, n_heads, head_dim) -> (B, T, C)
attn_output_concat = attn_output.transpose(1, 2).contiguous().view(B, T, C)
final_output = W_o(attn_output_concat)

print(f"\nConcatenated: {attn_output_concat.shape}")
print(f"Final output: {final_output.shape}")
print(f"Input shape:  {x.shape}")
print(f"Output shape: {final_output.shape}")
print("Input and output shapes match -- as expected.")

### Multi-Head Attention as a PyTorch Module

Now let us package everything into a clean `nn.Module`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        
        # Project to Q, K, V
        q = self.W_q(x)  # (B, T, C)
        k = self.W_k(x)
        v = self.W_v(x)
        
        # Reshape: (B, T, C) -> (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        if mask is not None:
            # mask shape: (T, T) or (1, 1, T, T)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(~mask, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        
        # Weighted sum of values
        attn_out = torch.matmul(attention_weights, v)  # (B, n_heads, T, head_dim)
        
        # Concatenate heads: (B, n_heads, T, head_dim) -> (B, T, C)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        
        # Output projection
        output = self.W_o(attn_out)
        
        return output, attention_weights


# Instantiate and test
torch.manual_seed(42)
mha = MultiHeadAttention(EMBED_DIM, N_HEADS)

x = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
causal = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool))

output, attn_weights = mha(x, mask=causal)

print(f"Input shape:            {x.shape}")
print(f"Output shape:           {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}  -- (batch, n_heads, T, T)")

# Parameter count
n_params = sum(p.numel() for p in mha.parameters())
print(f"\nTotal parameters: {n_params:,}")
print(f"  = 4 x (embed_dim x embed_dim) = 4 x {EMBED_DIM}x{EMBED_DIM} = {4 * EMBED_DIM * EMBED_DIM:,}")

---
## 6. Visualizations

### 6a. Attention Weight Heatmap -- Single Head

We visualize the attention pattern of head 0 for the first example in the batch, using token labels for clarity.

In [ ]:
# Single head attention heatmap with token labels
head_idx = 0
n_tokens_show = 8  # Show first 8 tokens for readability

w = attn_weights[0, head_idx, :n_tokens_show, :n_tokens_show].detach().numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(w, cmap='viridis', aspect='auto', vmin=0)

ax.set_xticks(range(n_tokens_show))
ax.set_yticks(range(n_tokens_show))
ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(TOKEN_LABELS_8, fontsize=11)
ax.set_xlabel('Key (attends to)', fontsize=12)
ax.set_ylabel('Query (from)', fontsize=12)
ax.set_title(f'Attention Weights -- Head {head_idx}', fontsize=14)

# Annotate cells with weight values
for i in range(n_tokens_show):
    for j in range(n_tokens_show):
        val = w[i, j]
        color = 'white' if val < 0.3 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, shrink=0.8, label='Attention weight')
plt.tight_layout()
plt.show()

**Analysis**: The lower-triangular pattern confirms causal masking is in effect. Notice how different query positions distribute their attention differently -- some focus heavily on one key position while others spread attention more evenly. These patterns would become more structured after training.

### 6b. All Heads Side by Side

Each head can learn a different attention pattern. Let us compare all four heads.

In [ ]:
fig, axes = plt.subplots(1, N_HEADS, figsize=(5 * N_HEADS, 5))

for h in range(N_HEADS):
    ax = axes[h]
    w = attn_weights[0, h, :n_tokens_show, :n_tokens_show].detach().numpy()
    im = ax.imshow(w, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_title(f'Head {h}', fontsize=13)
    ax.set_xticks(range(n_tokens_show))
    ax.set_yticks(range(n_tokens_show))
    ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=9)
    if h == 0:
        ax.set_yticklabels(TOKEN_LABELS_8, fontsize=9)
        ax.set_ylabel('Query', fontsize=11)
    else:
        ax.set_yticklabels([])
    ax.set_xlabel('Key', fontsize=11)

plt.suptitle('Attention Patterns Across All Heads', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Analysis**: Even with random (untrained) weights, the four heads already show distinct patterns. After training, we would expect specialization: some heads might focus on the immediately preceding token ("local" attention), while others attend to the first token or specific syntactic positions.

### 6c. Effect of Scaling: Histogram of Attention Weights

A histogram comparison that shows how scaling affects the distribution of attention weights.

In [ ]:
# Recompute attention with and without scaling for comparison
torch.manual_seed(42)
x_hist = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
q_h = mha.W_q(x_hist).view(BATCH_SIZE, SEQ_LEN, N_HEADS, HEAD_DIM).transpose(1, 2)
k_h = mha.W_k(x_hist).view(BATCH_SIZE, SEQ_LEN, N_HEADS, HEAD_DIM).transpose(1, 2)

scores_raw = torch.matmul(q_h, k_h.transpose(-2, -1))
causal_4d = causal.unsqueeze(0).unsqueeze(0)

# Without scaling
scores_no_scale = scores_raw.masked_fill(~causal_4d, float('-inf'))
weights_no_scale = F.softmax(scores_no_scale, dim=-1)

# With scaling
scores_with_scale = (scores_raw / (HEAD_DIM ** 0.5)).masked_fill(~causal_4d, float('-inf'))
weights_with_scale = F.softmax(scores_with_scale, dim=-1)

# Extract non-masked weights for histogram (lower triangle only)
tril_indices = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN)).bool()
vals_no_scale = weights_no_scale[0, 0][tril_indices].detach().numpy()
vals_with_scale = weights_with_scale[0, 0][tril_indices].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(vals_no_scale, bins=50, color='tomato', edgecolor='darkred', alpha=0.8)
axes[0].set_title('WITHOUT scaling', fontsize=13)
axes[0].set_xlabel('Attention weight', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].axvline(1.0 / SEQ_LEN, color='black', linestyle='--', label=f'Uniform = {1/SEQ_LEN:.3f}')
axes[0].legend(fontsize=10)

axes[1].hist(vals_with_scale, bins=50, color='steelblue', edgecolor='navy', alpha=0.8)
axes[1].set_title('WITH scaling (/ sqrt(d_k))', fontsize=13)
axes[1].set_xlabel('Attention weight', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].axvline(1.0 / SEQ_LEN, color='black', linestyle='--', label=f'Uniform = {1/SEQ_LEN:.3f}')
axes[1].legend(fontsize=10)

plt.suptitle('Distribution of Attention Weights', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Analysis**: Without scaling, the histogram is heavily bimodal: most weights cluster near 0 or 1, meaning the softmax is saturated. With scaling, the distribution is much more spread out, allowing the model to express nuanced, distributed attention patterns. This is crucial for learning.

---
## 7. Ablation Studies

We systematically remove each component to understand its contribution.

### 7a. Ablation: Remove Scaling

In [ ]:
# Ablation: attention without scaling
torch.manual_seed(42)
x_abl = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

# Helper function for entropy
def attention_entropy(weights):
    """Average entropy of attention distributions."""
    log_w = weights.clamp(min=1e-9).log()
    ent = -(weights * log_w).sum(dim=-1)  # (B, n_heads, T)
    return ent.mean().item()

# Get Q, K from the MHA module
q_abl = mha.W_q(x_abl).view(BATCH_SIZE, SEQ_LEN, N_HEADS, HEAD_DIM).transpose(1, 2)
k_abl = mha.W_k(x_abl).view(BATCH_SIZE, SEQ_LEN, N_HEADS, HEAD_DIM).transpose(1, 2)

# Unscaled scores
scores_unscaled = torch.matmul(q_abl, k_abl.transpose(-2, -1))
scores_unscaled = scores_unscaled.masked_fill(~causal_4d, float('-inf'))
weights_unscaled = F.softmax(scores_unscaled, dim=-1)

# Scaled scores
scores_scaled = torch.matmul(q_abl, k_abl.transpose(-2, -1)) / (HEAD_DIM ** 0.5)
scores_scaled = scores_scaled.masked_fill(~causal_4d, float('-inf'))
weights_scaled_abl = F.softmax(scores_scaled, dim=-1)

print("Average attention entropy:")
print(f"  Unscaled: {attention_entropy(weights_unscaled):.4f}")
print(f"  Scaled:   {attention_entropy(weights_scaled_abl):.4f}")
print(f"  Max possible (uniform over all {SEQ_LEN} positions): {np.log(SEQ_LEN):.4f}")

# Check how "one-hot" the unscaled weights are
max_unscaled = weights_unscaled.max(dim=-1).values.mean().item()
max_scaled = weights_scaled_abl.max(dim=-1).values.mean().item()
print(f"\nAverage max attention weight:")
print(f"  Unscaled: {max_unscaled:.4f} (closer to 1.0 = more one-hot)")
print(f"  Scaled:   {max_scaled:.4f}")

In [ ]:
# Visual comparison: unscaled vs scaled attention for head 0
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, w_tensor, title in [
    (axes[0], weights_unscaled, 'No Scaling (saturated softmax)'),
    (axes[1], weights_scaled_abl, 'With Scaling (healthy softmax)'),
]:
    w_np = w_tensor[0, 0, :n_tokens_show, :n_tokens_show].detach().numpy()
    im = ax.imshow(w_np, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_title(title, fontsize=13)
    ax.set_xticks(range(n_tokens_show))
    ax.set_yticks(range(n_tokens_show))
    ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(TOKEN_LABELS_8, fontsize=9)
    ax.set_xlabel('Key', fontsize=11)
    ax.set_ylabel('Query', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    # Annotate
    for i in range(n_tokens_show):
        for j in range(n_tokens_show):
            val = w_np[i, j]
            if val > 0.01:
                color = 'white' if val < 0.4 else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.suptitle('Ablation: Effect of Removing Scaling', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Analysis**: Without scaling, the attention weights are nearly one-hot -- almost all mass goes to a single key position. This is the softmax saturation problem. With scaling, attention is distributed across multiple positions, giving the model much richer representational capacity.

### 7b. Ablation: Remove Causal Mask

In [ ]:
# Run MHA with and without causal mask
torch.manual_seed(42)
x_mask_abl = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

output_causal, weights_causal = mha(x_mask_abl, mask=causal)
output_no_mask, weights_no_mask = mha(x_mask_abl, mask=None)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, w_tensor, title in [
    (axes[0], weights_causal, 'With Causal Mask'),
    (axes[1], weights_no_mask, 'Without Mask (information leakage!)'),
]:
    w_np = w_tensor[0, 0, :n_tokens_show, :n_tokens_show].detach().numpy()
    im = ax.imshow(w_np, cmap='viridis', aspect='auto', vmin=0)
    ax.set_title(title, fontsize=13)
    ax.set_xticks(range(n_tokens_show))
    ax.set_yticks(range(n_tokens_show))
    ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(TOKEN_LABELS_8, fontsize=9)
    ax.set_xlabel('Key', fontsize=11)
    ax.set_ylabel('Query', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Ablation: Effect of Removing Causal Mask', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the information leakage
print("Future attention mass (should be 0 for causal, >0 without mask):")
print()

future_mask = ~torch.tril(torch.ones(SEQ_LEN, SEQ_LEN)).bool()  # upper triangle = future

for name, w_tensor in [("With causal mask", weights_causal), ("Without mask", weights_no_mask)]:
    # Sum attention weights going to future positions
    w_batch0_head0 = w_tensor[0, 0]  # (T, T)
    future_total = w_batch0_head0[future_mask].sum().item()
    past_total = w_batch0_head0[~future_mask].sum().item()
    print(f"  {name}:")
    print(f"    Total attention to past+current: {past_total:.4f}")
    print(f"    Total attention to future:       {future_total:.4f}")
    print()

**Analysis**: Without the causal mask, significant attention mass flows to future positions. In a language model, this means the model could "cheat" by looking at the answer when predicting the next token. The causal mask is what makes GPT-2 autoregressive: each token's representation is built solely from past context.

### 7c. Ablation: Single Head vs Multi-Head

In [ ]:
# Compare single-head (n_heads=1) vs multi-head (n_heads=4)
torch.manual_seed(42)

mha_single = MultiHeadAttention(EMBED_DIM, n_heads=1)
mha_multi = MultiHeadAttention(EMBED_DIM, n_heads=4)

x_head_abl = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
causal_mask_bool = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool))

_, weights_single = mha_single(x_head_abl, mask=causal_mask_bool)
_, weights_multi = mha_multi(x_head_abl, mask=causal_mask_bool)

print(f"Single-head weights shape: {weights_single.shape}  -- 1 head, full {EMBED_DIM}-dim")
print(f"Multi-head weights shape:  {weights_multi.shape}  -- 4 heads, {HEAD_DIM}-dim each")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 4.5))

# Single head
w_single = weights_single[0, 0, :n_tokens_show, :n_tokens_show].detach().numpy()
im = axes[0].imshow(w_single, cmap='viridis', aspect='auto', vmin=0, vmax=1)
axes[0].set_title('Single Head\n(d=64)', fontsize=12)
axes[0].set_xticks(range(n_tokens_show))
axes[0].set_yticks(range(n_tokens_show))
axes[0].set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=8)
axes[0].set_yticklabels(TOKEN_LABELS_8, fontsize=8)
axes[0].set_ylabel('Query', fontsize=10)
axes[0].set_xlabel('Key', fontsize=10)

# Multi-head: show all 4 heads
for h in range(N_HEADS):
    ax = axes[h + 1]
    w_h = weights_multi[0, h, :n_tokens_show, :n_tokens_show].detach().numpy()
    im = ax.imshow(w_h, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_title(f'Multi-Head {h}\n(d={HEAD_DIM})', fontsize=12)
    ax.set_xticks(range(n_tokens_show))
    ax.set_yticks(range(n_tokens_show))
    ax.set_xticklabels(TOKEN_LABELS_8, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels([])
    ax.set_xlabel('Key', fontsize=10)

plt.suptitle('Single Head vs Multi-Head Attention Patterns', fontsize=15, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: diversity of attention patterns
def pattern_diversity(weights):
    """Measure how different attention patterns are across heads.
    Uses average pairwise L1 distance between head attention distributions.
    """
    n_h = weights.shape[1]
    if n_h == 1:
        return 0.0
    total = 0.0
    count = 0
    for i in range(n_h):
        for j in range(i + 1, n_h):
            diff = (weights[0, i] - weights[0, j]).abs().mean().item()
            total += diff
            count += 1
    return total / count

print(f"Pattern diversity (avg pairwise L1):")
print(f"  Single head:  {pattern_diversity(weights_single):.4f} (only 1 head, no diversity)")
print(f"  Multi-head:   {pattern_diversity(weights_multi):.4f}")

# Entropy comparison
ent_single = attention_entropy(weights_single)
ent_multi = attention_entropy(weights_multi)
print(f"\nAverage attention entropy:")
print(f"  Single head:  {ent_single:.4f}")
print(f"  Multi-head:   {ent_multi:.4f}")

**Analysis**: The single head produces one monolithic attention pattern, while the four multi-heads produce diverse patterns. This diversity is the key advantage of multi-head attention: the model can simultaneously attend to information at different positions and in different representational subspaces. In trained models, different heads specialize (e.g., syntactic vs. semantic relationships).

---
## 8. Putting It All Together: Full Forward Pass

Let us trace a complete forward pass through the `MultiHeadAttention` module, summarizing every transformation.

In [ ]:
torch.manual_seed(123)
mha_final = MultiHeadAttention(EMBED_DIM, N_HEADS)
x_final = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
causal_final = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool))

print("=== Full Multi-Head Attention Forward Pass ===")
print()

# Step 1: Input
print(f"1. Input x:          {x_final.shape}  (batch, seq_len, embed_dim)")

# Step 2: Linear projections
q = mha_final.W_q(x_final)
k = mha_final.W_k(x_final)
v = mha_final.W_v(x_final)
print(f"2. After W_q/W_k/W_v: {q.shape}  (same shape, different subspace)")

# Step 3: Reshape to heads
B, T, C = q.shape
q_h = q.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
k_h = k.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
v_h = v.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
print(f"3. Split into heads:  {q_h.shape}  (batch, n_heads, seq_len, head_dim)")

# Step 4: Attention scores
s = torch.matmul(q_h, k_h.transpose(-2, -1)) / (HEAD_DIM ** 0.5)
print(f"4. Attention scores:  {s.shape}  (batch, n_heads, seq_len, seq_len)")

# Step 5: Causal mask
s = s.masked_fill(~causal_final.unsqueeze(0).unsqueeze(0), float('-inf'))
print(f"5. After causal mask: {s.shape}  (future positions = -inf)")

# Step 6: Softmax
w = F.softmax(s, dim=-1)
print(f"6. Attention weights: {w.shape}  (softmax over keys)")

# Step 7: Weighted sum
o = torch.matmul(w, v_h)
print(f"7. Per-head output:   {o.shape}  (weighted sum of values)")

# Step 8: Concatenate
o_cat = o.transpose(1, 2).contiguous().view(B, T, C)
print(f"8. Concatenated:      {o_cat.shape}  (heads merged back)")

# Step 9: Output projection
final = mha_final.W_o(o_cat)
print(f"9. Final output:      {final.shape}  (after W_o projection)")
print()
print(f"Input shape == Output shape: {x_final.shape == final.shape}")

---
## 9. Gradient Flow Verification

A quick sanity check that gradients flow through the attention mechanism, which is essential for training.

---
## 10. Attention Weight Statistics Summary

In [ ]:
torch.manual_seed(42)
mha_grad = MultiHeadAttention(EMBED_DIM, N_HEADS)
x_grad = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM, requires_grad=True)

output_grad, _ = mha_grad(x_grad, mask=causal_final)
loss = output_grad.sum()
loss.backward()

print("Gradient check:")
print(f"  Input gradient shape:  {x_grad.grad.shape}")
print(f"  Input gradient norm:   {x_grad.grad.norm().item():.4f}")
print(f"  Input gradient mean:   {x_grad.grad.mean().item():.6f}")
print()

for name, param in mha_grad.named_parameters():
    if param.grad is not None:
        print(f"  {name:5s} gradient norm: {param.grad.norm().item():.4f}")

print("\nAll gradients are non-zero -- backpropagation works correctly.")

In [ ]:
# Summary statistics across different configurations
torch.manual_seed(42)
x_summary = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)

configs = [
    ("Multi-head (4), scaled, causal", MultiHeadAttention(EMBED_DIM, 4), causal_final),
    ("Multi-head (4), scaled, no mask", MultiHeadAttention(EMBED_DIM, 4), None),
    ("Single head, scaled, causal", MultiHeadAttention(EMBED_DIM, 1), causal_final),
]

print(f"{'Configuration':<40s} | {'Entropy':>8s} | {'Max wt':>8s} | {'Min wt':>10s}")
print("-" * 75)

for name, model, mask in configs:
    torch.manual_seed(42)  # Reset for reproducibility
    _, w_cfg = model(x_summary, mask=mask)
    ent = attention_entropy(w_cfg)
    mx = w_cfg.max().item()
    mn = w_cfg[w_cfg > 0].min().item() if (w_cfg > 0).any() else 0.0
    print(f"{name:<40s} | {ent:>8.4f} | {mx:>8.4f} | {mn:>10.6f}")

print(f"\nReference: uniform entropy for {SEQ_LEN} tokens = {np.log(SEQ_LEN):.4f}")

---
## Key Insights

### Core Mechanism
- **Attention** computes a weighted sum of value vectors, where the weights come from query-key dot products followed by softmax. This lets each token dynamically select what information to incorporate from other tokens.

### Scaling
- **Scaling by sqrt(d_k)** prevents softmax saturation. Without it, dot products grow with dimension, pushing softmax outputs toward one-hot vectors. This kills gradient flow and prevents the model from learning distributed attention patterns.

### Causal Masking
- **Causal masking** is what makes GPT-2 autoregressive. By setting future positions to `-inf` before softmax, we guarantee that each token can only attend to itself and previous tokens. This is essential for next-token prediction: the model must not see the answer it is trying to predict.

### Multi-Head Attention
- **Multiple heads** allow the model to attend to different types of information simultaneously. Each head operates in a lower-dimensional subspace (`head_dim = embed_dim / n_heads`), and the outputs are concatenated and projected back. In trained models, heads specialize: some track syntactic dependencies, others handle coreference, positional patterns, or semantic similarity.

### Computational Cost
- Attention is **O(T^2 * d)** in both time and memory, where T is sequence length. This quadratic scaling with sequence length is the main bottleneck for long-context models and motivates techniques like sparse attention, linear attention, and sliding-window attention.

### Connection to GPT-2
- GPT-2 uses this exact architecture: multi-head causal self-attention with scaled dot products. The only additions in the full transformer block are layer normalization, residual connections, and a feed-forward network -- the attention mechanism we built here is the core computational engine.